In [ ]:
pip install transformers pandas torch tqdm

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
import torch.nn as nn
from tqdm import tqdm
import os
from sklearn.metrics import accuracy_score

In [ ]:
# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        loss1 = loss2 = loss3 = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels).mean()

            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                loss2 = loss_fct(logits2[subjective_mask], level2_labels[subjective_mask]).mean()

                neutral_subjective_mask = (level2_labels == 0) & subjective_mask
                if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                    loss3 = loss_fct(logits3[neutral_subjective_mask], level3_labels[neutral_subjective_mask]).mean()

        total_loss = loss1 + loss2 + loss3
        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

            neutral_mask = (l1 == 2) & (l2 == 0)
            if neutral_mask.sum() > 0:
                preds3 = torch.argmax(logits3[neutral_mask], dim=1)
                all_l3_preds.extend(preds3.cpu().tolist())
                all_l3_labels.extend(l3[neutral_mask].cpu().tolist())


    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Validation Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")
    return l1_acc, l2_acc, l3_acc

# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            evaluate(model, val_loader, device=device)

        model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{epoch+1}.pt")
        optimizer_path = os.path.join(save_dir, f"optimizer_epoch{epoch+1}.pt")
        epoch_file = os.path.join(save_dir, "last_epoch.txt")

        torch.save(model.state_dict(), model_path)
        torch.save(optimizer.state_dict(), optimizer_path)
        with open(epoch_file, "w") as f:
            f.write(str(epoch+1))
        print(f"📦 Checkpoint saved for epoch {epoch+1}")

# ---------- Main Execution ----------
if __name__ == "__main__":
    df = pd.read_csv("/content/combined_train.csv")
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

    dataset = CryptoQADataset(df, tokenizer)
    val_size = int(0.1 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = HierarchicalClassifier()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "saved_models"
    epoch_file = os.path.join(save_dir, "last_epoch.txt")
    start_epoch = 0

    if os.path.exists(epoch_file):
        with open(epoch_file) as f:
            start_epoch = int(f.read())

        if start_epoch > 0:
            model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{start_epoch}.pt")
            optimizer_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")
            print(f"Resuming from Epoch {start_epoch}")
            model.load_state_dict(torch.load(model_path))
            optimizer.load_state_dict(torch.load(optimizer_path))

    train(model, train_loader, optimizer, epochs=5, start_epoch=start_epoch, val_loader=val_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Epoch 1: 100%|██████████| 724/724 [04:21<00:00,  2.77it/s]


🟢 Epoch 1 Training Loss: 1.7076
🧪 Running validation after epoch 1


Evaluating: 100%|██████████| 81/81 [00:09<00:00,  8.86it/s]


✅ Validation Accuracy - Level 1: 0.8054, Level 2: 0.7816, Level 3: 0.8579
📦 Checkpoint saved for epoch 1


Epoch 2: 100%|██████████| 724/724 [04:22<00:00,  2.76it/s]


...

✅ Validation Accuracy - Level 1: 0.8366, Level 2: 0.8000, Level 3: 0.8634
📦 Checkpoint saved for epoch 4


Epoch 5: 100%|██████████| 724/724 [04:22<00:00,  2.76it/s]


🟢 Epoch 5 Training Loss: 0.5076
🧪 Running validation after epoch 5


Evaluating: 100%|██████████| 81/81 [00:08<00:00,  9.12it/s]


✅ Validation Accuracy - Level 1: 0.8265, Level 2: 0.7988, Level 3: 0.8543
📦 Checkpoint saved for epoch 5


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['MAIN'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/reddit_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 32/32 [00:03<00:00,  8.03it/s]


✅ Test Accuracy - Level 1: 0.8980, Level 2: 0.8519, Level 3: 0.8585
Testing model from epoch 2


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.40it/s]


...

✅ Test Accuracy - Level 1: 0.8880, Level 2: 0.8571, Level 3: 0.8616
Testing model from epoch 4


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.31it/s]


✅ Test Accuracy - Level 1: 0.8780, Level 2: 0.8623, Level 3: 0.8585
Testing model from epoch 5


Testing: 100%|██████████| 32/32 [00:03<00:00,  8.80it/s]

✅ Test Accuracy - Level 1: 0.8860, Level 2: 0.8571, Level 3: 0.8616


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['Tweet'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/twitter_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.33it/s]


✅ Test Accuracy - Level 1: 0.6014, Level 2: 0.7120, Level 3: 0.6111
Testing model from epoch 2


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.73it/s]


...

✅ Test Accuracy - Level 1: 0.6900, Level 2: 0.7840, Level 3: 0.6889
Testing model from epoch 4


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.41it/s]


✅ Test Accuracy - Level 1: 0.6900, Level 2: 0.7680, Level 3: 0.7111
Testing model from epoch 5


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.25it/s]

✅ Test Accuracy - Level 1: 0.7110, Level 2: 0.7840, Level 3: 0.7111


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['comment'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/youtube_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.39it/s]


✅ Test Accuracy - Level 1: 0.9120, Level 2: 0.8325, Level 3: 0.9042
Testing model from epoch 2


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.61it/s]


...

✅ Test Accuracy - Level 1: 0.9000, Level 2: 0.8325, Level 3: 0.9208
Testing model from epoch 4


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.56it/s]


✅ Test Accuracy - Level 1: 0.9000, Level 2: 0.8517, Level 3: 0.8958
Testing model from epoch 5


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.41it/s]

✅ Test Accuracy - Level 1: 0.8960, Level 2: 0.8134, Level 3: 0.9042


In [ ]:
# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        loss1 = loss2 = loss3 = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels).mean()

            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                loss2 = loss_fct(logits2[subjective_mask], level2_labels[subjective_mask]).mean()

                neutral_subjective_mask = (level2_labels == 0) & subjective_mask
                if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                    loss3 = loss_fct(logits3[neutral_subjective_mask], level3_labels[neutral_subjective_mask]).mean()

        total_loss = loss1 + loss2 + loss3
        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

            neutral_mask = (l1 == 2) & (l2 == 0)
            if neutral_mask.sum() > 0:
                preds3 = torch.argmax(logits3[neutral_mask], dim=1)
                all_l3_preds.extend(preds3.cpu().tolist())
                all_l3_labels.extend(l3[neutral_mask].cpu().tolist())


    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Validation Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")
    return l1_acc, l2_acc, l3_acc

# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models_1', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            evaluate(model, val_loader, device=device)

        model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{epoch+1}.pt")
        optimizer_path = os.path.join(save_dir, f"optimizer_epoch{epoch+1}.pt")
        epoch_file = os.path.join(save_dir, "last_epoch.txt")

        torch.save(model.state_dict(), model_path)
        torch.save(optimizer.state_dict(), optimizer_path)
        with open(epoch_file, "w") as f:
            f.write(str(epoch+1))
        print(f"📦 Checkpoint saved for epoch {epoch+1}")

# ---------- Main Execution ----------
if __name__ == "__main__":
    df = pd.read_csv("/content/full_train.csv")
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

    dataset = CryptoQADataset(df, tokenizer)
    val_size = int(0.1 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = HierarchicalClassifier()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "saved_models_1"
    epoch_file = os.path.join(save_dir, "last_epoch.txt")
    start_epoch = 0

    if os.path.exists(epoch_file):
        with open(epoch_file) as f:
            start_epoch = int(f.read())

        if start_epoch > 0:
            model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{start_epoch}.pt")
            optimizer_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")
            print(f"Resuming from Epoch {start_epoch}")
            model.load_state_dict(torch.load(model_path))
            optimizer.load_state_dict(torch.load(optimizer_path))

    train(model, train_loader, optimizer, epochs=5, start_epoch=start_epoch, val_loader=val_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Epoch 1: 100%|██████████| 804/804 [04:52<00:00,  2.75it/s]


🟢 Epoch 1 Training Loss: 1.8092
🧪 Running validation after epoch 1


Evaluating: 100%|██████████| 90/90 [00:10<00:00,  8.92it/s]


✅ Validation Accuracy - Level 1: 0.6639, Level 2: 0.7208, Level 3: 0.5595
📦 Checkpoint saved for epoch 1


Epoch 2: 100%|██████████| 804/804 [04:51<00:00,  2.76it/s]


...

✅ Validation Accuracy - Level 1: 0.8473, Level 2: 0.8525, Level 3: 0.8649
📦 Checkpoint saved for epoch 4


Epoch 5: 100%|██████████| 804/804 [04:51<00:00,  2.76it/s]


🟢 Epoch 5 Training Loss: 0.6658
🧪 Running validation after epoch 5


Evaluating: 100%|██████████| 90/90 [00:09<00:00,  9.06it/s]


✅ Validation Accuracy - Level 1: 0.8347, Level 2: 0.8377, Level 3: 0.8605
📦 Checkpoint saved for epoch 5


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['comment'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/youtube_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models_1/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.61it/s]


✅ Test Accuracy - Level 1: 0.8360, Level 2: 0.5766, Level 3: 0.4208
Testing model from epoch 2


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.50it/s]


...

✅ Test Accuracy - Level 1: 0.9240, Level 2: 0.9211, Level 3: 0.9750
Testing model from epoch 4


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.87it/s]


✅ Test Accuracy - Level 1: 0.9600, Level 2: 0.9474, Level 3: 0.9750
Testing model from epoch 5


Testing: 100%|██████████| 32/32 [00:03<00:00,  9.72it/s]

✅ Test Accuracy - Level 1: 0.9780, Level 2: 0.9522, Level 3: 0.9875


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['MAIN'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/reddit_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models_1/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 32/32 [00:04<00:00,  7.41it/s]


✅ Test Accuracy - Level 1: 0.7620, Level 2: 0.8234, Level 3: 0.7736
Testing model from epoch 2


Testing: 100%|██████████| 32/32 [00:04<00:00,  7.25it/s]


...

✅ Test Accuracy - Level 1: 0.9320, Level 2: 0.8935, Level 3: 0.9119
Testing model from epoch 4


Testing: 100%|██████████| 32/32 [00:04<00:00,  7.86it/s]


✅ Test Accuracy - Level 1: 0.9500, Level 2: 0.9143, Level 3: 0.9434
Testing model from epoch 5


Testing: 100%|██████████| 32/32 [00:03<00:00,  8.07it/s]

✅ Test Accuracy - Level 1: 0.9400, Level 2: 0.9247, Level 3: 0.9717


In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['Tweet'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)
        return logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def test_model(model, dataloader, device='cuda'):
    model.to(device)
    model.eval()

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

        neutral_mask = (l1 == 2) & (l2 == 0)
        if neutral_mask.sum() > 0:
            preds3 = torch.argmax(logits3[neutral_mask], dim=1)
            all_l3_preds.extend(preds3.cpu().tolist())
            all_l3_labels.extend(l3[neutral_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Test Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")

# ---------- Main ----------
if __name__ == "__main__":
    df_test = pd.read_csv("/content/twitter_test.csv")  # Replace with your test dataset path
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)
    test_dataset = CryptoQADataset(df_test, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    for epoch in range(1, 6):  # Loop through epochs 1 to 5
        print(f"Testing model from epoch {epoch}")
        model = HierarchicalClassifier()
        checkpoint_path = f"saved_models_1/bertweet_hierarchical_epoch{epoch}.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location='cuda' if torch.cuda.is_available() else 'cpu'))
        test_model(model, test_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Testing model from epoch 1


Testing: 100%|██████████| 27/27 [00:03<00:00,  8.43it/s]


✅ Test Accuracy - Level 1: 0.2984, Level 2: 0.7200, Level 3: 0.1444
Testing model from epoch 2


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.19it/s]


...

✅ Test Accuracy - Level 1: 0.7413, Level 2: 0.7920, Level 3: 0.8111
Testing model from epoch 4


Testing: 100%|██████████| 27/27 [00:03<00:00,  8.73it/s]


✅ Test Accuracy - Level 1: 0.7925, Level 2: 0.8320, Level 3: 0.8333
Testing model from epoch 5


Testing: 100%|██████████| 27/27 [00:02<00:00,  9.40it/s]

✅ Test Accuracy - Level 1: 0.7902, Level 2: 0.8800, Level 3: 0.9000


In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from tqdm import tqdm
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Model ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        return logits1, logits2, logits3

# ---------- Dataset ----------
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze()
        }

# ---------- Predict Function ----------
def predict(model, dataloader):
    model.eval()
    model.to(device)

    preds_level1 = []
    preds_level2 = []
    preds_level3 = []

    for batch in tqdm(dataloader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits1, logits2, logits3 = model(input_ids, attention_mask)

        level1 = torch.argmax(logits1, dim=1)
        level2 = torch.full_like(level1, -1)
        level3 = torch.full_like(level1, -1)

        subjective_mask = level1 == 2
        if subjective_mask.sum() > 0:
            sub_logits2 = logits2[subjective_mask]
            level2_preds = torch.argmax(sub_logits2, dim=1)
            level2[subjective_mask] = level2_preds

            neutral_mask = (level2 == 0) & subjective_mask
            if neutral_mask.sum() > 0:
                sub_logits3 = logits3[neutral_mask]
                level3_preds = torch.argmax(sub_logits3, dim=1)
                level3[neutral_mask] = level3_preds

        preds_level1.extend(level1.cpu().tolist())
        preds_level2.extend(level2.cpu().tolist())
        preds_level3.extend(level3.cpu().tolist())

    return preds_level1, preds_level2, preds_level3

# ---------- Load Model ----------
model = HierarchicalClassifier()
model.load_state_dict(torch.load("saved_models_1/bertweet_hierarchical_epoch5.pt", map_location=device))

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

# ---------- Reddit ----------
df_reddit = pd.read_csv("/content/CRYPTO_REDDIT_TEST.csv")
reddit_texts = df_reddit['title'].fillna('') + " " + df_reddit['selftext'].fillna('')
reddit_dataset = InferenceDataset(reddit_texts.tolist(), tokenizer)
reddit_loader = DataLoader(reddit_dataset, batch_size=16)

r1, r2, r3 = predict(model, reddit_loader)
df_reddit["level 1"] = r1
df_reddit["level 2"] = r2
df_reddit["level 3"] = r3
df_reddit = df_reddit[["title", "selftext", "MAIN", "level 1", "level 2", "level 3"]]
df_reddit.to_csv("crypto_test_reddit.csv", index=False)

# ---------- Twitter ----------
df_twitter = pd.read_csv("/content/CRYPTO_TWITTER_TEST.csv")
twitter_dataset = InferenceDataset(df_twitter['Text'].tolist(), tokenizer)
twitter_loader = DataLoader(twitter_dataset, batch_size=16)

t1, t2, t3 = predict(model, twitter_loader)
df_twitter["Level 1"] = t1
df_twitter["Level 2"] = t2
df_twitter["Level 3"] = t3
df_twitter = df_twitter[["Text", "Level 1", "Level 2", "Level 3"]]
df_twitter.to_csv("crypto_test_tweet.csv", index=False)

# ---------- YouTube ----------
df_yt = pd.read_csv("/content/CRYPTO_YOUTUBE_TEST.csv")
yt_dataset = InferenceDataset(df_yt['MAIN'].tolist(), tokenizer)
yt_loader = DataLoader(yt_dataset, batch_size=16)

y1, y2, y3 = predict(model, yt_loader)
df_yt["Level 1"] = y1
df_yt["Level 2"] = y2
df_yt["Level 3"] = y3
df_yt = df_yt[["comment_id", "MAIN", "Level 1", "Level 2", "Level 3"]]
df_yt.to_csv("crypto_test_youtube.csv", index=False)

# ---------- Zip Results ----------
import zipfile
with zipfile.ZipFile("submission.zip", "w") as zipf:
    zipf.write("crypto_test_reddit.csv")
    zipf.write("crypto_test_tweet.csv")
    zipf.write("crypto_test_youtube.csv")

print("🎉 Submission zip created: submission.zip")


emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Predicting: 100%|██████████| 32/32 [00:03<00:00,  9.84it/s]

🎉 Submission zip created: submission.zip


In [ ]:
# ---------- Dataset Definition ----------
class CryptoQADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['text'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if level1 == 2 else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if (level1 == 2 and level2 == 0) else -1, dtype=torch.long)
        }

# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="vinai/bertweet-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)
        self.classifier2 = nn.Linear(hidden_size, 3)
        self.classifier3 = nn.Linear(hidden_size, 4)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(reduction='none')
        loss1 = loss2 = loss3 = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels).mean()

            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                loss2 = loss_fct(logits2[subjective_mask], level2_labels[subjective_mask]).mean()

                neutral_subjective_mask = (level2_labels == 0) & subjective_mask
                if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                    loss3 = loss_fct(logits3[neutral_subjective_mask], level3_labels[neutral_subjective_mask]).mean()

        total_loss = loss1 + loss2 + loss3
        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        subjective_mask = (l1 == 2)
        if subjective_mask.sum() > 0:
            preds2 = torch.argmax(logits2[subjective_mask], dim=1)
            all_l2_preds.extend(preds2.cpu().tolist())
            all_l2_labels.extend(l2[subjective_mask].cpu().tolist())

            neutral_mask = (l1 == 2) & (l2 == 0)
            if neutral_mask.sum() > 0:
                preds3 = torch.argmax(logits3[neutral_mask], dim=1)
                all_l3_preds.extend(preds3.cpu().tolist())
                all_l3_labels.extend(l3[neutral_mask].cpu().tolist())


    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0

    print(f"✅ Validation Accuracy - Level 1: {l1_acc:.4f}, Level 2: {l2_acc:.4f}, Level 3: {l3_acc:.4f}")
    return l1_acc, l2_acc, l3_acc

# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models_1', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            evaluate(model, val_loader, device=device)

        model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{epoch+1}.pt")
        optimizer_path = os.path.join(save_dir, f"optimizer_epoch{epoch+1}.pt")
        epoch_file = os.path.join(save_dir, "last_epoch.txt")

        torch.save(model.state_dict(), model_path)
        torch.save(optimizer.state_dict(), optimizer_path)
        with open(epoch_file, "w") as f:
            f.write(str(epoch+1))
        print(f"📦 Checkpoint saved for epoch {epoch+1}")

# ---------- Main Execution ----------
if __name__ == "__main__":
    df = pd.read_csv("/content/full_train.csv")
    tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", use_fast=True)

    dataset = CryptoQADataset(df, tokenizer)
    val_size = int(0.1 * len(dataset))
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

    model = HierarchicalClassifier()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "/content/"
    epoch_file = os.path.join(save_dir, "last_epoch.txt")
    start_epoch = 0

    os.makedirs(save_dir, exist_ok=True) # Ensure the save directory exists

    if os.path.exists(epoch_file):
        with open(epoch_file) as f:
            start_epoch = int(f.read())

        if start_epoch > 0:
            model_path = os.path.join(save_dir, f"bertweet_hierarchical_epoch{start_epoch}.pt")
            optimizer_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")
            print(f"Resuming from Epoch {start_epoch}")
            # Check if the files exist before loading
            if os.path.exists(model_path) and os.path.exists(optimizer_path):
                model.load_state_dict(torch.load(model_path))
                optimizer.load_state_dict(torch.load(optimizer_path))
            else:
                print(f"Warning: Checkpoint files for epoch {start_epoch} not found. Starting training from epoch 0.")
                start_epoch = 0


    train(model, train_loader, optimizer, epochs=10, start_epoch=start_epoch, val_loader=val_loader)

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Resuming from Epoch 5


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory